<div style="padding: 60px 40px; border-radius: 12px;  text-align: center;">

<h1 style="font-size: 2.4em; font-weight: 800; margin-bottom: 16px; letter-spacing: -0.5px;">LLM-for-LLM</h1>
<h2 style="font-size: 1.4em; font-weight: 400; color: #1a6bb5; margin-bottom: 32px;">用智能体 AI 优化关键基础模型工作负载</h2>
<h2 style="font-size: 1.2em; font-weight: 400; color: #1a1a1a; margin-bottom: 32px;">Vincent Fang · AMD AIG</h2>
<hr style="border-color: rgba(0,0,0,0.15); margin: 24px 0;">
<h5 style="color: #555555; font-size: 1.05em;">平台：AMD ROCm &nbsp;&middot;&nbsp; 运行时：vLLM &nbsp;&middot;&nbsp; GPU：Radeon™ RX 7900 XTX · Radeon 开发者云(radeon.anruicloud.com) </p>
</div>

## 工作坊路线图

智能体 AI 正在成为推理优化里的实用工具。强模型已经具备不少关于 GPU kernel、运行时、性能分析器和调优流程的知识。现在，在 ROCm 和 AMD GPU 上，这些知识可以端到端地应用到真实的模型服务栈中。

| # | 主题 | 形式 |
|---|------|------|
| 1 | 为什么智能体优化正在变得可行 | 开场背景 |
| 2 | AMD ROCm + vLLM 推理栈是什么样的 | 架构讲解 |
| 3 | 方法：用 `vllm-optimize` 做智能体优化 | 工作流概览 |
| 4 | 智能体能从性能分析轨迹中真正发现什么 | 真实数据 |
| 5 | TunableOps 如何离线选择 GEMM kernel | 技术深挖 |
| 6 | Qwen3-8B 的端到端优化结果 | 基准测试图表 |
| **7** | **现场演示：让智能体实时优化一个模型** | **OpenCode** |


---
## 1 · 为什么智能体优化正在变得可行

### 一个具体结果

在这个工作坊中，智能体把 Qwen3-8B 在中等并发下的吞吐从 **356 TPS 提升到 565 TPS**。

> **结果：吞吐提升 +58.4%，生成每个 token 的成本约降低 37%。**

假设一个模型服务每天处理 **100 万次请求**，每次请求生成 **1024 个输出 token**：

| 场景 | 吞吐提升 | 每 token 成本 | 粗略年节省* |
|------|----------|---------------|-------------|
| 常见调优收益 | +10% | 约降低 9% | 约 $2.6K |
| 较强调优收益 | +50% | 约降低 33% | 约 $13K |
| **本工作坊结果** | **+58%** | **约降低 37%** | **约 $15K** |

<small>*示例估算按 $3/hr 的 GPU 价格计算。重点是每 token 成本下降，而不是具体云价格。</small>

### 为什么这个工作流重要

瓶颈横跨 vLLM batching、GEMM 调度、ROCm 库以及 AMD GPU 架构。想要安全地拿到收益，不能靠猜，而要基于真实工作负载里的证据：基准测试结果、性能分析轨迹、实际 GEMM shape，以及端到端验证。

> 这个工作坊展示的是：LLM 智能体如何在 ROCm 上跑通这个闭环：**测量 -> 分析 -> 调优 -> 验证**。


---
## 2 · vLLM + ROCm 推理架构

vLLM 原生支持 ROCm，因此开发者可以在 AMD GPU 上继续使用熟悉的模型服务 API 和高级推理能力，覆盖从 Instinct 数据中心加速卡到 Radeon 开发者工作站的不同环境。

![vLLM + ROCm 推理架构](assets/vllm_rocm_architecture_fresh.png)

**要点：** vLLM 提供模型服务抽象，ROCm 把这个抽象连接到 AMD 软件库和 GPU 硬件。本工作坊的优化工作都发生在这个栈里，并且以真实工作负载证据为依据。


---
## 3 · 方法：`vllm-optimize` Skill

`vllm-optimize` 把性能优化流程封装成一个可重复执行的闭环：对模型做基准测试、采集证据、针对实测瓶颈调优、验证结果，并留下开发者可以检查的报告。

<div style="margin: 18px 0 8px; padding: 28px 30px; border-radius: 18px; background: linear-gradient(135deg, #f7fbff 0%, #eef6ff 100%); border: 1px solid #d8e8f7;">
  <div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 18px;">
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #5aa7d6; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #39779f; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 0</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">环境检查</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">识别 GPU、ROCm 和 vLLM 版本</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 10 秒</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #62bf8e; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #3b8b64; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 1</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">服务启动</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">启动 vLLM，并开启性能分析器与 TunableOps 记录</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 2 分钟</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #f4a64b; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #b66a14; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 2</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">基准测试 + 性能分析</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">执行并发扫描并采集轨迹</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 12-15 分钟</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #7c8ee6; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #5867c9; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 3</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">分析</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">定位 GPU 时间瓶颈和优化目标</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 1 分钟</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #b476bf; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #8b4d96; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 4</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Kernel 优化</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">TunedOps / Triton Optimize / GEAK Agent</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 10 分钟</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #e36b6b; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #bf4747; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">阶段 5-6</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">验证 + 报告</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">注入调优后的 kernel，执行端到端检查并汇总结果</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">约 8 分钟 + 报告</div>
    </div>
  </div>
</div>

**产出：** 从基线测量到优化后的服务配置，约 **35 分钟**，同时留下可复查的运行产物和最终报告。


---
## 4 · 找到瓶颈

基线运行结束后，智能体会先对服务工作负载做性能分析，并问一个最关键的问题：**GPU 时间到底花到哪里了？**

在这次 Qwen3-8B、`conc=16` 的运行中，decode 阶段主要被 GEMM kernel 占用。因此，第一优化目标就是 GEMM，而不是靠猜测去调一些无关的 flag 或栈里的其他部分。

### GPU Kernel 时间分布 — Qwen3-8B，conc=16，decode 阶段


![GPU kernel time breakdown](assets/part4_gpu_kernel_breakdown.png)


### 诊断

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 14px 0 8px;">
  <div style="background:#fff5f5; border:1px solid #ffd7d7; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#c53030;">91%</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">GPU 活跃时间</div>
    <div style="color:#5b6678; margin-top:6px;">消耗在 GEMM kernel 上</div>
  </div>
  <div style="background:#f5f9ff; border:1px solid #d7e7ff; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#2b6cb0;">decode</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">阶段瓶颈</div>
    <div style="color:#5b6678; margin-top:6px;">在中等并发下清晰可见</div>
  </div>
  <div style="background:#f7fff8; border:1px solid #cbe8d0; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#2f855a;">GEMM</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">优化目标</div>
    <div style="color:#5b6678; margin-top:6px;">下一阶段就聚焦这里</div>
  </div>
</div>

这里最重要的不是某个固定结论，而是工作流本身：智能体先用性能分析证据缩小问题范围，再开始做优化。


---
## 5 · 解决瓶颈

一旦确认 GEMM 是主要成本，智能体就进入 kernel 级优化路径。在这个例子里，第一步是用 **TunableOps 做离线 GEMM kernel 选择**。

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 16px 0 18px;">
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #5aa7d6; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#1d4f73;">1 · 记录</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">捕获真实服务流量中出现的 GEMM 调用。</div>
  </div>
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #b476bf; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#743a80;">2 · 调优</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">离线测试候选 rocBLAS / hipBLASLt kernel。</div>
  </div>
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #62bf8e; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#2f6f4e;">3 · 注入</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">在服务启动时加载选出的 kernel，不修改 vLLM 源码。</div>
  </div>
</div>

除了 TunableOps，同一套工作流也可以扩展到 **Triton Kernel Agent**，用于更复杂的算子融合；也可以按需启用 **GEAK**，做更高级的 HIP kernel 生成。这些路径背后的设计原则一致：优化能力要可插拔，并且只围绕实测瓶颈展开，**不改 vLLM 源码，也不改模型结构**。

### Kernel 级优化结果


![Kernel optimization result](assets/part5_kernel_optimize_result.png)


---
## 6 · 衡量收益

最后一步是检查端到端服务吞吐。微基准里的提升只有在集成后仍能改善真实 vLLM 工作负载，才算真正有价值。

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 14px 0 18px;">
  <div style="background:#f6f8ff; border:1px solid #dce5ff; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#2b6cb0;">356 TPS</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">基线</div>
    <div style="color:#5b6678; margin-top:6px;">conc=16</div>
  </div>
  <div style="background:#f4fff7; border:1px solid #ccebd6; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#2f855a;">565 TPS</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">优化后</div>
    <div style="color:#5b6678; margin-top:6px;">同一工作负载</div>
  </div>
  <div style="background:#fff8ef; border:1px solid #f7d7ad; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#c05621;">+58.4%</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">吞吐</div>
    <div style="color:#5b6678; margin-top:6px;">每 token 成本约降低 37%</div>
  </div>
</div>

### 基准测试配置

| 项目 | 设置 |
|------|------|
| 模型 | Qwen3-8B（`hidden=4096`，`layers=36`，`heads=32`，`intermediate=12288`） |
| GPU | AMD RDNA3 gfx1100 · 48 WGP / 96 CU · GDDR6 |
| vLLM | `--enforce-eager`，TP=1，`max_model_len=4096` |
| dtype | bfloat16 |
| 输入 / 输出长度 | ISL=1024，OSL=1024 tokens |


![End-to-end throughput result](assets/part6_e2e_throughput.png)


---
## 7 · 现场演示

<div style=" padding: 32px; border-radius: 10px; ">

<h3 style="color: #1a6bb5; margin-top: 0;">动手实践：自己跑一遍完整流程</h3>

<div style="background: rgba(0,0,0,0.03); border: 1px solid rgba(0,100,200,0.2); border-radius: 12px; padding: 18px 22px; margin: 18px 0 22px;">
  <div style="display: grid; grid-template-columns: 160px 1fr; row-gap: 12px; column-gap: 18px; align-items: baseline;">
    <div style="color: #1a6bb5; font-weight: 700;">环境</div>
    <div style="color: #1a1a1a;">无锡 Radeon Pro 集群</div>
    <div style="color: #1a6bb5; font-weight: 700;">容量</div>
    <div style="color: #1a1a1a;">支持 160+ 并发用户</div>
    <div style="color: #1a6bb5; font-weight: 700;">访问地址</div>
    <div style="color: #1a5faa; font-weight: 700; word-break: break-all;">https://radeon.anruicloud.com/</div>
  </div>
</div>

<p style="color: #333333;">最后一部分不只是看演示。目标是让每个人都能启动一个 ROCm 就绪的实例，端到端跑完整个流程，并在每个阶段边界检查留下来的证据。</p>

<h4 style="color: #1a6bb5;">建议流程：</h4>

<ol style="color: #1a1a1a; line-height: 2;">
<li>使用ModelScope账号登录<code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">https://radeon.anruicloud.com/</code>，从应用库中选择 <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">Agentic Inference Optimization</code></li>
<li>安装 vLLM Optimize 技能：<code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">bash install.sh</code></li>
<li>在终端中打开 OpenCode：<code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">opencode</code></li>
<li>选择一个智能体模型，例如 <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">/models Claude Sonnet 4.6</code></li>
<li>调用技能，例如 <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">/vllm-optimize qwen/Qwen3-8B</code> 或 <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">/vllm-optimize qwen/Qwen3.5-4B</code></li>
<li>开启演示模式，并在每个阶段完成后查看证据</li>
<li>熟悉流程后，可以换成自己的模型或配置再跑一遍</li>
</ol>

</div>